In [ ]:
import json
import os

os.chdir("..")

In [2]:
from cluster_intrep_repo.utils import DOMAIN_PHRASES

In [8]:
# def make_mean_reprs_layer(layer: int):
all_representations = {}

for i in range(1, 16):
    with open(f"multilayer_representations/multilayer_7k/mystery_{i}/mean_reprs_mystery_{i}_multi_layer.json") as f:
        all_representations[f"mystery_{i}"] = json.load(f)

In [9]:
import numpy as np

avg_representations = {k: {} for k in all_representations.keys()}

def process_layer(layer: int):
    representations = {
        k: v[f"{layer}"] for k, v in all_representations.items()
    }
    
    action_reprs = {
        k: {
            kk: np.array(vv) - np.array(v["mean_actions"]) for kk, vv in v["mean_reprs"].items() if kk in DOMAIN_PHRASES[k]["actions"].values()
        } for k, v in representations.items()
    }

    predicate_reprs = {
        k: {
            kk: np.array(vv) - np.array(v["mean_predicates"]) for kk, vv in v["mean_reprs"].items() if kk in DOMAIN_PHRASES[k]["predicates"].values()
        } for k, v in representations.items()
    }

    # action_reprs = {
    #     k: {
    #         kk: np.array(vv) for kk, vv in v["mean_reprs"].items() if kk in DOMAIN_PHRASES[k]["actions"].values()
    #     } for k, v in representations.items()
    # }

    # predicate_reprs = {
    #     k: {
    #         kk: np.array(vv) for kk, vv in v["mean_reprs"].items() if kk in DOMAIN_PHRASES[k]["predicates"].values()
    #     } for k, v in representations.items()
    # }
    
    reverse_phrases = {
        k: {kk: {
            vvv: kkk for kkk, vvv in vv.items()
        } for kk, vv in v.items() }
        for k, v in DOMAIN_PHRASES.items()
    }
    
    predicates = list(reverse_phrases["mystery_3"]["predicates"].values())
    actions = list(reverse_phrases["mystery_3"]["actions"].values())
    
    action_reprs = {
        k: {
            reverse_phrases[k]["actions"][kk]: vv for kk, vv in v.items()
        } for k, v in action_reprs.items()
    }

    predicate_reprs = {
        k: {
            reverse_phrases[k]["predicates"][kk]: vv for kk, vv in v.items()
        } for k, v in predicate_reprs.items()
    }
    
    mean_action_reprs = {
        action: np.mean(
            np.stack(
                [action_reprs[m][action] for m in action_reprs]
            ), axis=0
        ) for action in actions
    }
    
    mean_predicate_reprs = {
        pr: np.mean(
            np.stack(
                [predicate_reprs[m][pr] for m in predicate_reprs]
            ), axis=0
        ) for pr in predicates
    }
    
    for i in range(1, 16):
        reprs = {
            "mean_domain": [0] * 5120,
            "mean_actions": [0] * 5120,
            "mean_predicates": [0] * 5120,
            "mean_reprs": {
                DOMAIN_PHRASES[f"mystery_{i}"]["actions"][action]: mean_action_reprs[action].tolist() for action in actions
            }
        }
        
        reprs["mean_reprs"].update({
            DOMAIN_PHRASES[f"mystery_{i}"]["predicates"][predicate]: mean_predicate_reprs[predicate].tolist() for predicate in predicates
        })
            
        avg_representations[f"mystery_{i}"][f"{layer}"] = reprs

In [10]:
from tqdm import tqdm

for i in tqdm(range(50)):
    process_layer(i)

100%|██████████| 50/50 [00:03<00:00, 14.77it/s]


In [11]:
avg_representations["mystery_1"]["2"]["mean_reprs"]["attack"]

[0.04972127278645833,
 -0.00677490234375,
 0.039920806884765625,
 -0.0779937744140625,
 -0.035672760009765624,
 -0.019447835286458333,
 -0.020126851399739583,
 0.005475870768229167,
 0.021183013916015625,
 0.00440673828125,
 0.06577657063802084,
 -0.001171875,
 0.020178858439127603,
 -0.003361002604166667,
 0.10491689046223958,
 0.019051361083984374,
 0.01775188446044922,
 -0.010504659016927083,
 0.0024883270263671873,
 0.04093297322591146,
 -0.06983709335327148,
 -0.06804962158203125,
 -0.011124674479166667,
 0.05432815551757812,
 0.07435302734375,
 -0.006818517049153646,
 -0.03198153177897135,
 -0.046025594075520836,
 -0.03451334635416667,
 0.029327392578125,
 -0.00732421875,
 -0.047306219736735024,
 0.015529378255208334,
 -0.015649922688802085,
 0.03397369384765625,
 -0.03738199869791667,
 0.0214111328125,
 0.07447560628255208,
 -0.24043858846028646,
 0.1011016845703125,
 -0.0107818603515625,
 0.017093149820963542,
 -0.05321057637532552,
 0.0012125651041666667,
 -0.00522143046061197

In [12]:
from pathlib import Path

for i in range(1, 16):
    path = Path(f"multilayer_representations_avg/multilayer_7k/mystery_{i}/mean_reprs_mystery_{i}_multi_layer.json")
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(avg_representations[f"mystery_{i}"], f, indent=4)

: 